# 2b. ERA5: model evaluation and observed change (figures)

Everything here is drawn from what notebook 02a saved, so nothing slow runs. 02a explains each test and has a quick look at each step.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.patches import Patch

from plotting_modules import core, formatting, maps, timeseries
from extant import era5_evaluation as ev, paths
from extant.datatree import skip_empty, tree_to_dataset
from extant.logger import setup_logging
from extant.plots import era5_evaluation as evp, observed_change as ocp, response_change as rcp

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

## Settings

In [ ]:
#(t): As in 02a
VARIABLE = 'tas'
UNITS = '°C'
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
YEARS = slice(1979, 2013)
POINT = dict(lat=-82.5, lon=120.0)
POINT_LABEL = formatting.format_latlon(POINT)

#(t): The model and season of the one-model, one-season figures, and the region and model of part 2's
MODEL = 'CanESM5'
SEASON = 'DJF'
OBS_REGION = 'Antarctica (90–65°S)'
OBS_MODEL = 'CanESM5'

In [ ]:
#(t): Where 02a saved a result
def result_path(name):
    return paths.result(name, VARIABLE, RUN)

## Which results

Run **one** of the next two sections: they say whether to open the sample's results or the full data's.

### Sample data

In [ ]:
RUN = 'sample'

### Full data (JASMIN)

In [ ]:
RUN = 'full'

## Open the results

First where each one is (`paths.find`: DATA_DIR first, then SCRATCH), then each opened.

In [ ]:
#(t): The results of 02a used here
RESULTS = ['first_look', 'historical_point', 'histnat_point', 'era5_point', 'moment_test', 'statistic_test',
           'raw_ranks', 'anomaly_ranks', 'detrended_ranks', 'rank_anatomy_data', 'rank_anatomy_test', 'eval_table',
           'demo_tests', 'demo_anomaly_ranks', 'demo_detrended_ranks', 'demo_plume', 'eval_map_table',
           'spatial_table', 'region_tests', 'record_curves', 'point_record_curves', 'detections', 'scaling_ant_nat',
           'scaling_single_forcings']
result_paths = {name: paths.find(result_path(name)) for name in RESULTS}
result_paths

In [ ]:
#(t): ERA5's 1979-2013 mean in SEASON, and each model's ensemble mean minus it
first_look_ds = xr.open_zarr(result_paths['first_look']).load()

In [ ]:
#(t): At the point: each model's historical and hist-nat members (/<model>/<experiment>), and ERA5
historical_point_tree = xr.open_datatree(result_paths['historical_point'], engine='zarr').load()
histnat_point_tree = xr.open_datatree(result_paths['histnat_point'], engine='zarr').load()
era5_point_da = xr.open_zarr(result_paths['era5_point'])['tas'].load()
historical_point_tree

In [ ]:
#(t): The tests at the point, /<model>/historical: the moments, the other statistics, and the rank histograms
moment_test_tree = xr.open_datatree(result_paths['moment_test'], engine='zarr').load()
statistic_test_tree = xr.open_datatree(result_paths['statistic_test'], engine='zarr').load()
raw_ranks_tree = xr.open_datatree(result_paths['raw_ranks'], engine='zarr').load()
anomaly_ranks_tree = xr.open_datatree(result_paths['anomaly_ranks'], engine='zarr').load()
detrended_ranks_tree = xr.open_datatree(result_paths['detrended_ranks'], engine='zarr').load()
moment_test_tree

In [ ]:
#(t): Every test's percentile and verdict, on (model, statistic, season)
eval_table_ds = xr.open_zarr(result_paths['eval_table']).load()

In [ ]:
#(t): The rank test step by step: MODEL's anomalies in SEASON, and the test
rank_anatomy_ds = xr.open_zarr(result_paths['rank_anatomy_data']).load()
anatomy_result_ds = xr.open_zarr(result_paths['rank_anatomy_test']).load()

In [ ]:
#(t): The toy ensembles
demo_tests_ds = xr.open_zarr(result_paths['demo_tests']).load()
demo_anomaly_ranks_ds = xr.open_zarr(result_paths['demo_anomaly_ranks']).load()
demo_detrended_ranks_ds = xr.open_zarr(result_paths['demo_detrended_ranks']).load()
demo_plume_ds = xr.open_zarr(result_paths['demo_plume']).load()

In [ ]:
#(t): The maps: the moment test and the field test, and the spatial evaluation
eval_map_table_ds = xr.open_zarr(result_paths['eval_map_table']).load()
spatial_table_ds = xr.open_zarr(result_paths['spatial_table']).load()

In [ ]:
#(t): Part 2: ERA5 among each experiment in OBS_REGION (/<model>/<experiment>), the record curves, the detection
#(t): maps and the scaling factors
region_tests_tree = xr.open_datatree(result_paths['region_tests'], engine='zarr').load()
record_curves_ds = xr.open_zarr(result_paths['record_curves']).load()
point_curves_ds = xr.open_zarr(result_paths['point_record_curves']).load()
detections_ds = xr.open_zarr(result_paths['detections']).load()
scaling_ant_nat_ds = xr.open_zarr(result_paths['scaling_ant_nat']).load()
scaling_single_ds = xr.open_zarr(result_paths['scaling_single_forcings']).load()

## A first look

ERA5's 1979–2013 mean in `SEASON`, and each model's historical ensemble mean minus it: the mean-state bias, which only the tests of raw values see.

In [ ]:
era5_mean_da = first_look_ds['era5_mean']
panels = maps.polar_grid(era5_mean_da, levels=np.linspace(float(era5_mean_da.min()), float(era5_mean_da.max()), 15),
                         cmap='RdYlBu_r', title=f'ERA5, {SEASON} 1979-2013', cbar_label=f'Temperature ({UNITS})', tag=False)
panels = maps.polar_grid(first_look_ds['model_bias'], col_dim='model', levels=np.linspace(-5, 5, 11), cmap='RdBu_r',
                         title=f'Model minus ERA5, {SEASON} 1979-2013 (ensemble mean)', cbar_label=f'Bias ({UNITS})',
                         tag=False)

## 1. How the tests work: toy ensembles with a known answer

Before trusting the tests on real data, here they are on synthetic ensembles where we *know* what is wrong. 02a's `ev.simulate_ensemble` makes 50 members and a pseudo-ERA5, 1979–2013, four seasons, each series `trend × decades + noise × N(0, 1)`. Pseudo-ERA5 always has a trend of 0.3 per decade and noise 1; each scenario (`ev.SCENARIOS`) changes one thing about the model:

| Scenario | The model | What the tests should say |
|---|---|---|
| Consistent | identical to ERA5 | nothing flagged, apart from ~10% by chance |
| Too little noise | noise 0.6x | ERA5 often outside the plume; U-shaped histograms; std flagged high |
| Too much noise | noise 1.6x | ERA5 never leaves the plume; dome-shaped histograms; std flagged low |
| Signal too strong | trend 0.9/decade | U shape for anomalies but **flat once detrended**; trend and rank trend flagged |
| Mean-state bias | 2° too warm | raw plume offset; only the mean flagged; every anomaly test passes |

Columns: the raw time series with the plume; the rank histograms of anomalies and of detrended anomalies (bars: pseudo-ERA5; grey bars: the range a perfect model gives); ERA5's percentile for each statistic (✓ inside the grey 5–95% band, ✗ outside).

In [ ]:
panels = evp.scenario_grid(demo_plume_ds, demo_anomaly_ranks_ds, demo_detrended_ranks_ds, demo_tests_ds,
                           season=SEASON, title='The tests on toy ensembles with a known answer')

**Reading the figure**
- **Signal vs noise.** Compare the two rank-histogram columns. A noise problem (rows 2–3) shows in both. A signal problem (row 4) shows only for anomalies: a trend mismatch pushes ERA5 out of the ensemble at both ends of the record, which looks like a U, and detrending removes it.
- **Raw values vs anomalies.** A mean-state bias (row 5) puts ERA5 outside the raw plume much of the time, but once each series is measured from its own mean, every test passes. This is the "raw values don't fit but the anomalies do" case.
- **Too little noise can also flag the mean.** A too-narrow ensemble underestimates how far a 35-year mean can wander. So read the noise tests first.

**Calibration**

Over 200 random toy worlds (40 members), a perfect model is flagged 6–12% of the time for every statistic (nominally 10%). Each fault is caught by its intended test almost every time:

| | noise too small | noise too large | signal too strong | mean bias |
|---|---|---|---|---|
| std (detrended) | 99% | 97% | 10% | 10% |
| width, Q95 − Q05 | 96% | 92% | – | – |
| cold / warm tail | 76% / 77% | 57% / 59% | – | – |
| rank dispersion (anomalies) | 100% | 100% | 68% | 6% |
| rank dispersion (detrended) | 100% | 100% | 8% | 8% |
| trend | 35% | 2% | 96% | 12% |
| rank trend | 21% | 4% | 100% | 9% |
| mean | 30% | 2% | 100% | 100% |

The tail tests are the weakest: with ~35 values per season, each tail rests on about two of them. A ✓ for a tail therefore means "no evidence of a problem", which is weaker than a ✓ for the width or std.

## 2. Distributions: do the raw values fit? do the anomalies?

For each season: each member's KDE (thin), the pooled ensemble (thick blue) and ERA5's (black), with ERA5's values as a rug; the ✓/✗ is the moment test of the mean (raw values) or of the standard deviation (anomalies). The Q–Q panel: the members' quantiles against ERA5's; on the 1:1 line the distributions match. ERA5 and every member share one kernel bandwidth, so differences come from the data rather than the smoothing.

In [ ]:
#(t): MODEL's members and ERA5 at the point
eval_members_da = historical_point_tree[MODEL]['historical']['tas']
eval_era5_da = era5_point_da

In [ ]:
#(t): Their KDEs and quantiles, for the figure (ev.distribution_summary): raw values, anomalies, detrended
eval_distributions = ev.distribution_summary(eval_members_da, eval_era5_da, ('raw', 'anomaly', 'detrended'))
eval_distributions['anomaly']

In [ ]:
#(t): One model, every season
panels = evp.distribution_grid(
    eval_distributions, moments=moment_test_tree[MODEL]['historical'].to_dataset(),
    units=UNITS, title=f'Distributions: {MODEL} historical vs ERA5 at {POINT_LABEL}',
)

In [ ]:
#(t): Every model's (raw values and anomalies), by model
all_distributions = {node.parent.name: ev.distribution_summary(node['tas'], era5_point_da, ('raw', 'anomaly'))
                     for node in historical_point_tree.leaves}

In [ ]:
#(t): Every model, one season
panels = evp.distribution_grid(all_distributions, season=SEASON, moments=moment_test_tree, units=UNITS,
                               title=f'Distributions, {SEASON}: every model vs ERA5 at {POINT_LABEL}')

In [ ]:
#(t): Detrended anomalies instead, to compare the noise alone
panels = evp.distribution_grid(
    eval_distributions, moments=moment_test_tree[MODEL]['historical'].to_dataset(),
    columns=(('kde', 'raw'), ('kde', 'detrended'), ('qq', 'detrended')),
    units=UNITS, title=f'Distributions (detrended): {MODEL} historical vs ERA5 at {POINT_LABEL}',
)

**What to look for**
- A raw-value KDE shifted from the ensemble is a mean-state bias. If the anomaly KDE then sits on top of the ensemble, the bias was the only problem.
- An anomaly KDE narrower or wider than the members' means the model has too much or too little variability. Check whether the std test (✓/✗) agrees.
- With ~35 values per season, a single ERA5 KDE is noisy. Judge it against the spread of the thin member curves, which have the same sample size, not against the smooth pooled curve.

## 3. Time series: is ERA5 inside or outside the ensemble plume?

Blue bands are the ensemble's central 75% (12.5–87.5%, as in Suarez-Gutierrez et al., 2021), 5–95% and full range each year, with its median. ERA5 is black, with an orange ▲/▼ wherever it is above or below *every* member. If ERA5 were a member, that would happen 200 / (N + 1) % of the time (a few %), as noted in each panel. In the anomaly column, green outlines show the hist-nat ensemble for context.

In [ ]:
#(t): MODEL's plume, raw values: the members' quantiles and range each year, and where ERA5 is (ev.ensemble_plume)
eval_raw_plume_ds = ev.ensemble_plume(eval_members_da, eval_era5_da, treatment='raw')

In [ ]:
#(t): ... and anomalies
eval_anomaly_plume_ds = ev.ensemble_plume(eval_members_da, eval_era5_da, treatment='anomaly')

In [ ]:
#(t): The two along a treatment dim, on (treatment, quantile, year, season)
point_plume_ds = xr.concat([eval_raw_plume_ds, eval_anomaly_plume_ds], dim=xr.Variable('treatment', ['raw', 'anomaly']))
point_plume_ds

In [ ]:
#(t): The historical ensemble-mean climatology at the point: the mean over members and years, per season
point_hist_climatology_da = eval_members_da.mean(['member', 'year'])
point_hist_climatology_da

In [ ]:
#(t): ERA5's climatology at the point, over the same years and missing-data mask
point_era5_climatology_da = eval_era5_da.mean('year')
point_era5_climatology_da

In [ ]:
#(t): Quick look: the raw-value plume's quantiles (colours) and ERA5 (black), one panel per season
point_raw_plume_ds = point_plume_ds.sel(treatment='raw', season=SEASONS)
fg = point_raw_plume_ds['quantiles'].plot(x='year', hue='quantile', col='season')
for ax, season in zip(fg.axs.flat, SEASONS):
    point_raw_plume_ds['obs'].sel(season=season).plot(ax=ax, color='k')

In [ ]:
#(t): Rows: raw values and anomalies; columns: seasons. Dashed lines: the two climatologies, on the raw values
panels = evp.plume_grid(point_plume_ds, row_dim='treatment', col_dim='season', units=UNITS,
                        title=f'Ensemble plume and ERA5, {VARIABLE}: {MODEL} {'historical'} at {POINT_LABEL}')
for ax, season in zip(panels.axes[0], SEASONS):
    ax.axhline(point_hist_climatology_da.sel(season=season), color=evp.ENSEMBLE['median'], lw=1.2, ls='--',
               label=f'{'historical'} ensemble-mean climatology')
    ax.axhline(point_era5_climatology_da.sel(season=season), color=evp.ERA5_COLOR, lw=1.2, ls='--',
               label='ERA5 climatology')
panels.axes[0, -1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)

In [ ]:
#(t): Each season's middle month, as a fraction of a year
#(c): DJF is labelled by its December's year, so its middle month (January) is in the next year
SEASON_MIDDLE = xr.DataArray([12.5, 3.5, 6.5, 9.5], coords={'season': ['DJF', 'MAM', 'JJA', 'SON']}) / 12


def seasons_in_time(obj):
    """Put (year, season) on one time axis, in decimal years at each season's middle month.

    year and season stay as coordinates along time, so a seasonal climatology can be
    looked up at every time step with .sel(season=...), like the old day-of-year lookup.
    """
    decimal_year_da = obj['year'] + SEASON_MIDDLE.sel(season=obj['season'])
    with_time = obj.assign_coords(time=decimal_year_da)
    stacked = with_time.stack(sample=['year', 'season'])
    stacked = stacked.reset_index('sample')        # year and season become plain coordinates
    stacked = stacked.swap_dims(sample='time')
    return stacked.sortby('time')

In [ ]:
#(t): The plume: 0-100% in 5% steps, drawn as nested bands from the widest (0-100%) in
PLUME_QUANTILES = np.round(np.linspace(0, 1, 21), 2)
PLUME_PAIRS = np.column_stack([PLUME_QUANTILES[:10], PLUME_QUANTILES[::-1][:10]])
PLUME_PAIRS

In [ ]:
#(t): MODEL's plume and ERA5 at the point, raw values, on (quantile, year, season)
point_raw_plume_ds = ev.ensemble_plume(eval_members_da, eval_era5_da, treatment='raw', quantiles=PLUME_QUANTILES)

#(t): The same for anomalies: each member and ERA5 minus its own mean in each season
point_anomaly_plume_ds = ev.ensemble_plume(eval_members_da, eval_era5_da, treatment='anomaly', quantiles=PLUME_QUANTILES)
point_anomaly_plume_ds

In [ ]:
#(t): The historical ensemble-mean climatology at the point: the mean over members and years, per season
point_hist_climatology_da = eval_members_da.mean(['member', 'year'])
#(t): ERA5's climatology at the point, over the same years and missing-data mask
point_era5_climatology_da = eval_era5_da.mean('year')
point_era5_climatology_da

In [ ]:
#(t): The raw-value plume on one time axis
point_raw_plume_time_ds = seasons_in_time(point_raw_plume_ds)
#(t): The anomaly plume on one time axis
point_anomaly_plume_time_ds = seasons_in_time(point_anomaly_plume_ds)
point_anomaly_plume_time_ds

In [ ]:
#(t): The historical climatology at every time step, looked up by season
point_hist_climatology_time_da = point_hist_climatology_da.sel(season=point_raw_plume_time_ds['season'])
#(t): ERA5's climatology at every time step, looked up by season
point_era5_climatology_time_da = point_era5_climatology_da.sel(season=point_raw_plume_time_ds['season'])
point_hist_climatology_time_da

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 8), layout='constrained', sharex=True)

timeseries.draw_plume(axes[0], point_raw_plume_time_ds['quantiles'], PLUME_PAIRS, median=None, color='C0', x_dim='time')
axes[0].plot(point_raw_plume_time_ds['time'], point_raw_plume_time_ds['obs'], color='k', lw=1, label='ERA5')
axes[0].plot(point_hist_climatology_time_da['time'], point_hist_climatology_time_da, color='C3', lw=1.2, ls='--', alpha=0.8,
             label=f'{'historical'} ensemble-mean climatology')
axes[0].plot(point_era5_climatology_time_da['time'], point_era5_climatology_time_da, color='orange', lw=1.2, ls='--', alpha=0.8,
             label='ERA5 climatology')
axes[0].set_title(f'Seasonal-mean {VARIABLE}', loc='left')
axes[0].set_ylabel(f'Seasonal-mean {VARIABLE} ({UNITS})')

timeseries.draw_plume(axes[1], point_anomaly_plume_time_ds['quantiles'], PLUME_PAIRS, median=None, color='C0', x_dim='time')
axes[1].plot(point_anomaly_plume_time_ds['time'], point_anomaly_plume_time_ds['obs'], color='k', lw=1)
axes[1].set_title("Anomalies from each series' own seasonal climatology", loc='left')
axes[1].set_ylabel(f'Anomaly ({UNITS})')
axes[1].set_xlabel('Year')

for ax in axes:
    core.style_ax(ax, tight_x=True)
    ax.set_xlim(1980, 1985)

plume_patch = Patch(color='C0', alpha=0.6, label=f'{MODEL} {'historical'} members (0–100% in 5% bands)')
handles = [plume_patch, *axes[0].get_legend_handles_labels()[0]]
fig.legend(handles=handles, loc='outside lower center', ncols=2, frameon=False)
fig.suptitle(f'Ensemble plume vs ERA5, {VARIABLE} ({POINT_LABEL})')

In [ ]:
#(t): hist-nat's anomaly plume at the point, as context: ERA5 should follow historical, not hist-nat
histnat_plume_ds = ev.ensemble_plume(histnat_point_tree[MODEL]['hist-nat']['tas'], era5_point_da, treatment='anomaly')

In [ ]:
#(t): One model: seasons x (raw values, anomalies), with hist-nat (green) for context
panels = evp.plume_grid(
    point_plume_ds, row_dim='season', col_dim='treatment', context=histnat_plume_ds.expand_dims(treatment=['anomaly']),
    units=UNITS, title=f'Ensemble plume and ERA5: {MODEL} historical at {POINT_LABEL}',
)

In [ ]:
#(t): ev.ensemble_plume of the anomalies on one model's node
@skip_empty
def anomaly_plume(ds):
    return ev.ensemble_plume(ds['tas'], era5_point_da, treatment='anomaly')

In [ ]:
#(t): Every model's anomaly plume; without the members, so they fit side by side on (model, quantile, year, season)
all_plumes_tree = historical_point_tree.map_over_datasets(anomaly_plume)
all_plumes_ds = tree_to_dataset(all_plumes_tree, dims=['model', 'experiment']).sel(experiment='historical', drop=True)

In [ ]:
panels = evp.plume_grid(all_plumes_ds, row_dim='model', col_dim='season', sharey=True, units=UNITS,
                        title=f"Anomalies: every model's plume and ERA5 at {POINT_LABEL}")

In [ ]:
#(t): % of seasons with ERA5 outside each model's range (bars), against chance, 200 / (N + 1) % (black)
outside_ds = ev.outside_frequency(all_plumes_ds)
outside_df = outside_ds['outside'].sel(season=SEASONS).transpose('model', 'season').to_pandas()
ax = outside_df.plot.bar(rot=0, ylabel='% of seasons outside the members')
ax.plot(range(len(outside_df)), outside_ds['expected_outside'].to_pandas().values, 'k_', ms=30, mew=2);

**What to look for**
- Raw values outside the plume in most years, with anomalies inside it: a mean-state bias only.
- Anomalies outside the plume *at both extremes* more often than chance: too little noise, or a signal problem (which one is settled in section 5).
- ERA5 drifting from one edge of the anomaly plume towards the other through the record: the trend differs (the signal).

## 4. The moment test, and the other statistics

Each panel is one statistic in one season: every member's value is a blue dot, the shading the members' 5–95%, ERA5 the black line and diamond. If the model were perfect, ERA5 would be just another dot; it is flagged (✗) when it falls outside the shading.

In [ ]:
#(t): MODEL's tests
eval_moments_ds = moment_test_tree[MODEL]['historical'].to_dataset()
eval_statistics_ds = statistic_test_tree[MODEL]['historical'].to_dataset()

In [ ]:
panels = evp.statistic_grid(eval_moments_ds, title=f'Moment test: {MODEL} historical vs ERA5 at {POINT_LABEL}')

In [ ]:
#(t): The signal and noise statistics (the quantities the response figures use), and lag-1 memory
panels = evp.statistic_grid(eval_statistics_ds,
                            statistics=['trend', 'emergence', 'width', 'lower_tail', 'upper_tail', 'lag1'],
                            title=f'Signal, noise and memory: {MODEL} historical vs ERA5 at {POINT_LABEL}')

In [ ]:
#(t): Record highs and lows (with no change at all, 1 + 1/2 + ... + 1/35 ≈ 4.1 of each are expected)
panels = evp.statistic_grid(eval_statistics_ds, statistics=['records_high', 'records_low'],
                            title=f'Records: {MODEL} historical vs ERA5 at {POINT_LABEL}')

## 5. Rank histograms (Suarez-Gutierrez et al., 2021)

02a goes through the test step by step. Bars are ERA5's rank frequencies; the dashed lines bound the pseudo-observations', the range a perfect model's histogram falls in. A U shape means too little spread (or a wrong trend pushing ERA5 out at both ends), a dome too much. The same U in anomalies and detrended anomalies is a noise problem; a U in anomalies only is a signal problem.

In [ ]:
panels = evp.rank_anatomy(rank_anatomy_ds['members'], rank_anatomy_ds['era5'], anatomy_result_ds, units=UNITS,
                          title=f'How the rank-histogram test works: {MODEL}, {SEASON} anomalies at {POINT_LABEL}')

In [ ]:
#(t): All four seasons pooled: raw values, anomalies, detrended
panels = evp.rank_histogram_row(
    {'Raw values': raw_ranks_tree[MODEL]['historical'].to_dataset(),
     'Anomalies': anomaly_ranks_tree[MODEL]['historical'].to_dataset(),
     'Detrended anomalies': detrended_ranks_tree[MODEL]['historical'].to_dataset()},
    title=f'Rank histograms, all seasons pooled: {MODEL} historical vs ERA5 at {POINT_LABEL}',
)

In [ ]:
#(t): Every model, compact (ranks grouped into 10 bins of rank / N so ensembles of any size line up)
panels = evp.rank_histogram_grid(
    {'Raw values': raw_ranks_tree, 'Anomalies': anomaly_ranks_tree, 'Detrended anomalies': detrended_ranks_tree},
    title=f'Rank histograms, all seasons pooled: every model vs ERA5 at {POINT_LABEL}',
)

## 6. All models: scorecard and signal-vs-noise diagram

**Scorecard.** Colour is ERA5's percentile among that model's members: blue, ERA5 below most members; red, above. Outlined cells are flagged (p < 0.1). Expect about one cell in ten flagged by chance: trust patterns over single cells.

**Signal vs noise.** ERA5's percentile for the trend (x: the signal) against the standard deviation of detrended anomalies (y: the noise). Inside the grey square the model is consistent on both.

In [ ]:
SCORECARD_STATISTICS = ['mean', 'std', 'skewness', 'kurtosis', 'trend', 'emergence', 'rank_trend', 'width',
                        'lower_tail', 'upper_tail', 'dispersion (anomaly)', 'dispersion (detrended)']
panels = evp.scorecard(eval_table_ds, statistics=SCORECARD_STATISTICS,
                       title=f"Where ERA5 falls among each model's members ({POINT_LABEL})")

In [ ]:
panels = evp.signal_noise_diagram(eval_table_ds, title=f'Signal and noise: trend vs interannual variability ({POINT_LABEL})')

In [ ]:
panels = evp.signal_noise_diagram(eval_table_ds, y='width',
                                  title=f'Signal and noise: trend vs distribution width ({POINT_LABEL})')

## 7. Maps: the moment test at every grid point

ERA5's percentile among each model's members, darkest where it is beyond their 5–95%. Top left of each panel, the field test: the % of the map where ERA5 is flagged, and the most a perfect model gives 90% of the time; ✗ when ERA5 is flagged over more. Models with fewer than 20 members cannot flag anything and are left blank.

In [ ]:
for statistic in ['mean', 'std', 'skewness', 'kurtosis']:
    panels = evp.percentile_maps(eval_map_table_ds, statistic)

In [ ]:
#(t): The field test for every model, moment and season: 1 (✗) where ERA5 is flagged over more of the map than a
#(t): perfect model gives, 0 (✓) where not, blank with too few members
#(c): The model names are Python objects after tree_to_dataset; xarray only puts strings on an axis
field_verdict_da = eval_map_table_ds['field_verdict'].sel(season=SEASONS)
field_verdict_da = field_verdict_da.assign_coords(model=field_verdict_da['model'].astype(str))
fg = field_verdict_da.plot(x='season', y='model', col='statistic', vmin=0, vmax=1, cmap='Reds', add_colorbar=False)

In [ ]:
#(t): How many models are consistent with ERA5 (ERA5 inside their members' 5-95%) at each grid point
#(c): Only models with enough members to be tested (N >= 20) are counted, as consistent or not
testable_da = eval_map_table_ds['testable'] == 1
n_eval_models = int(testable_da.any([dim for dim in testable_da.dims if dim != 'model']).sum())
count_cmap, count_norm = rcp.count_colormap(n_eval_models, base_cmap='Blues')
n_consistent_da = ((eval_map_table_ds['verdict'] == 0) & testable_da).sum('model')
panels = maps.polar_grid(
    n_consistent_da.sel(season=SEASONS),
    row_dim='statistic', col_dim='season',
    levels=np.arange(-0.5, n_eval_models + 1.5), cmap=count_cmap, norm=count_norm,
    label_fmt={'statistic': lambda s: ev.MOMENTS[s].label},
    title='Number of models consistent with ERA5',
    cbar_label="Models with ERA5 inside their members' 5-95% range",
    tag=False,
)
panels.extras['cbar'].set_ticks(range(n_eval_models + 1))

## 8. Where each model is adequate: the spatial evaluation of Suarez-Gutierrez et al. (2021)

The paper's grid-point evaluation (their sections 2.2.2 and 3.2–3.4), at every grid point and season. It does not use the members-as-observations rank histograms of section 5 (the paper keeps those for the global mean). Instead it reduces each point's rank histogram to three numbers and compares them with what a perfect model gives, using fixed thresholds (`ev.spatial_evaluation`):

1. **Anomalies.** Every member and ERA5 minus its own mean over 1979–2013, so a constant offset is gone and the forced change and the variability are left. (The paper uses 1961–1990, which is before ERA5 starts.)
2. **Years.** Only years ERA5 has; a point with fewer than 10 is not tested.
3. **Rank.** Each year, is ERA5 below every member (rank 0), above every member (rank N), or in between?
4. **Three percentages.** The % of years below every member, above every member, and inside the members' central 75% (12.5–87.5%).
5. **Against a perfect model.** If ERA5 were one more member, it would be below every member in 100 / (N + 1) % of years (2% for 50 members), above in as many, and in the central 75% about 75% of the time. Short records push these up by chance, so the paper flags only **≥ 10% below**, **≥ 10% above** or **> 80% in the middle**. Models with N < 20 are not tested with these thresholds: 10 members already put ERA5 beyond them 9% of the time on each side. `SPATIAL_THRESHOLDS = 'perfect_model'` (not in the paper) sets the thresholds from the members instead: each member in turn is treated as ERA5 against the other N − 1, and a percentage counts as a problem above the 95th percentile of theirs. That works for any N.
6. **Diagnosis**, using when the edge years fall (1979–1996 or 1997–2013):

| ERA5 is … | Diagnosis |
|---|---|
| below the members mostly late, or above them mostly early | the model **warms too much** (forced response too strong) |
| above the members mostly late, or below them mostly early | the model **warms too little** |
| both below and above, throughout | **too little variability**: the spread is too narrow |
| in the central 75% more than 80% of the time | **too much variability**: the spread is too wide |
| beyond one edge only, throughout | **one tail too short**: the shape of the distribution |
| none of these | **adequate** |

   The sign flips with time because anomalies are taken over the whole record: a model that warms too fast looks too cold early and too warm late, relative to ERA5. With both edges beyond 10%, both must point the same way to count as the forced response.
7. **Early against late.** A model's variability changes little over time, but the forcing does. So each problem over the whole record is checked for an uneven split between 1979–1996 and 1997–2013 (a binomial test, p < 0.1). Spread evenly, it is the **variability**; concentrated in one period, or flipping between them, the **forced response**. Each half is only ~17 years per season, too short to compare its own flags directly, which in 17 years would flag even a perfect model's middle about a third of the time.
8. **Score.** The adequate share of the tested area per model (each panel's corner), and how many models are adequate at each grid point (the paper's Fig. 8).

In [ ]:
#(t): The paper's Fig. 5: ERA5 beyond every member (red above, blue below) or too often in their middle (hatched)
panels = evp.spatial_maps(spatial_table_ds)

In [ ]:
#(t): What is wrong at each grid point (step 6)
panels = evp.diagnosis_maps(spatial_table_ds)

In [ ]:
#(t): Each problem early vs late: the same (variability) or changing (forced response)?
panels = evp.diagnosis_maps(spatial_table_ds, variable='period_diagnosis')

In [ ]:
#(t): The score: % of the tested area where each model is adequate, per season
ev.adequate_area(spatial_table_ds).sel(season=SEASONS).to_pandas().plot.bar(rot=0, ylabel='% of the area adequate');

In [ ]:
#(t): The paper's Fig. 8: how many models are adequate at each grid point
panels = evp.adequate_count_maps(ev.adequate_count(spatial_table_ds))

**Reading these maps**
- A model warming too much shows blue (ERA5 below every member) in the late period and red early, and a forced-response verdict for the period split. Too little variability shows as purple (both edges) with a variability verdict.
- Hatching (too much variability) is the least robust signal with ~35 years per season: the paper notes that short periods put the observations in the middle of the ensemble too often by chance.
- Each grid point is tested on ~35 values, so expect some flags by chance; read coherent regions. A regional mean (as in the observed-change section) is the more robust check, as the paper recommends.

# Part 2: observed change

## 1. Which experiments is the observed change consistent with?

Each row is one experiment: its members' 1979–2013 trends (dots) and their 5–95% range (bar); the black line is ERA5's. Where the line crosses a row's bar (✓), that experiment could have produced the observed trend. ERA5 outside **hist-nat** is a detection of forced change; ERA5 inside **historical** is consistency.

In [ ]:
panels = ocp.experiment_strips(region_tests_tree[OBS_MODEL], 'trend',
                               title=f'Observed trend vs each experiment: {OBS_MODEL}, {OBS_REGION}')

In [ ]:
#(t): The same for the S/N against itself (smoothed change over the record / its own noise)
panels = ocp.experiment_strips(region_tests_tree[OBS_MODEL], 'emergence',
                               title=f'Observed S/N vs each experiment: {OBS_MODEL}, {OBS_REGION}')

In [ ]:
#(t): One node's trend verdicts only (no members)
@skip_empty
def trend_verdicts(ds):
    return ds[['percentile', 'verdict', 'testable']].sel(statistic='trend', drop=True)

In [ ]:
#(t): Every model, the experiments as the scorecard's columns
trend_table_ds = tree_to_dataset(region_tests_tree.map_over_datasets(trend_verdicts), dims=['model', 'experiment'])
trend_table_ds = trend_table_ds.rename(experiment='statistic')
panels = evp.scorecard(trend_table_ds, title=f"ERA5's 1979-2013 trend among each experiment's members ({OBS_REGION})")

## 2. Record rates

Cumulative record highs and lows through 1979–2013, with the four seasons pooled so there are enough records to compare. The dashed curve is what a climate with no change gives (1 + 1/2 + … + 1/*n* per season). Record highs arriving faster than that, and faster than the hist-nat members', mean warming the natural range cannot produce; record lows arriving slower mean the cold end has moved too.

In [ ]:
panels = ocp.record_rates(record_curves_ds, title=f'Record rates, all seasons pooled: {OBS_MODEL}, {OBS_REGION}')

In [ ]:
#(t): At the point (much noisier than a regional mean)
panels = ocp.record_rates(point_curves_ds, title=f'Record rates, all seasons pooled: {OBS_MODEL}, {POINT_LABEL}')

## 3. Detection maps

The trend test at every grid point: detected (outside hist-nat's 5–95%) and consistent (inside historical's). At p < 0.1 about 10% of points land outside a range by chance; read coherent regions.

In [ ]:
panels = ocp.detection_maps(detections_ds)

In [ ]:
#(t): How many models call each point detected and consistent
panels = ocp.detection_count_maps(detections_ds)

## 4. Scaling factors: how much of each forced response is in the observations

β = 1 (the dashed line): the model's response has the observed size; an interval clear of 0 is a detection. The grey tick is the perfect-model median, which shows any bias in the method; it should be close to 1.

In [ ]:
#(t): Two signals: anthropogenic and natural
panels = ocp.scaling_factor_plot(scaling_ant_nat_ds, title='Scaling factors: anthropogenic (ANT) and natural (NAT)')
panels = ocp.attributable_trend_plot(scaling_ant_nat_ds, title='How much of the observed trend ANT and NAT explain')

In [ ]:
#(t): Four signals: greenhouse gases, aerosols, ozone and natural
panels = ocp.scaling_factor_plot(scaling_single_ds, title='Scaling factors: single forcings')
panels = ocp.attributable_trend_plot(scaling_single_ds, title='How much of the observed trend each forcing explains')